# Agent Yuzik - Запуск у Google Colab

Гэты нататнік дазваляе запусціць лакальнага AI-агента Юзік з падтрымкай TTS (тэкст-у-голас) на GPU у Google Colab.

**Важна:** Пераканайцеся, што ў вас уключаны GPU (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
import os

# Устаўце ваш ключ ад Google Gemini API сюды:
GEMINI_API_KEY = "УСТАЎЦЕ_СЮДЫ_ВАШ_КЛЮЧ"

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
os.environ["PORT"] = "7861"

print("✅ Ключы і налады паспяхова дададзены!")

In [ ]:
print("1/4: Усталёўка гукавых пакетаў і кланіраванне...\n")
!apt-get install -y espeak-ng libsndfile1 > /dev/null 2>&1

import os
if not os.path.exists("Agent-Yuzik"):
    !git clone https://github.com/tuteishygpt/Agent-Yuzik.git

%cd Agent-Yuzik

print("2/4: Усталёўка Python залежнасцяў (можа заняць 2-3 хвіліны)...\n")
!pip install -r requirements.txt > /dev/null 2>&1
!pip install fal_client uvicorn fastapi python-multipart > /dev/null 2>&1

print("✅ Залежнасці ўсталяваны!")

In [ ]:
print("3/4: Зборка вэб-інтэрфейсу...\n")
%cd frontend
!npm install > /dev/null 2>&1
!chmod +x node_modules/.bin/vite
!npm run build > /dev/null 2>&1
%cd ..
print("✅ Фронтэнд сабраны!")

In [ ]:
import os
import time
import threading

print("4/4: Запуск сервера і тунэля...\n")
def start_server():
    os.system("python app.py > server_log.txt 2>&1")

t = threading.Thread(target=start_server)
t.daemon = True
t.start()

time.sleep(10) # чакаем запуску каля 10 сек

print("🔥 ВАША СПАСЫЛКА НА ЮЗІКА 🔥")
print("---------------------------------------------------------")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!cloudflared tunnel --url http://127.0.0.1:7861
